# 🔧 Notebook 2: Data Preprocessing

**Môn học:** Học Máy (CO3001) — Học kỳ I, Năm học 2025–2026  
**Trường:** Đại học Bách Khoa, ĐHQG-HCM  
**Giảng viên hướng dẫn:** TS. Lê Thành Sách  
**Nhóm:** 13

---

### 🎯 Mục tiêu Notebook này
- Xử lý missing values: **Mean / Median / KNN Imputer**
- Encoding categorical features: **One-Hot / Ordinal**
- Feature scaling: **Standard / MinMax / Robust**
- So sánh các cấu hình preprocessing bằng **Logistic Regression probe**
- Lưu preprocessor, features (`.npy`) và train/test raw CSV để notebook sau dùng

---
**📋 Thứ tự chạy:** `01_EDA` → **`02_Preprocessing`** → `03_Traditional_ML` → `04_Deep_Learning`  
> ⚠️ Cần chạy `01_EDA` ít nhất một lần trước để tải dữ liệu về `data/raw/`.

## 1. 📦 Import Thư Viện & Cấu Hình Đường Dẫn

In [ ]:
import subprocess, sys
for pkg in ['category_encoders', 'lightgbm', 'xgboost']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)
print('✅ Cài đặt hoàn tất!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, urllib.request, joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'figure.dpi': 100})

# ── Cấu hình đường dẫn (giống main_notebook) ──────────────────────────────
_NB_DIR  = os.path.dirname(os.path.abspath('__file__'))  # notebooks/
ROOT_DIR = os.path.abspath(os.path.join(_NB_DIR, '..'))  # project root

DATA_DIR      = os.path.join(ROOT_DIR, 'data')
RAW_DATA_DIR  = os.path.join(DATA_DIR, 'raw')
PROC_DATA_DIR = os.path.join(DATA_DIR, 'processed')
FEAT_DIR      = os.path.join(ROOT_DIR, 'features')
MOD_DIR       = os.path.join(ROOT_DIR, 'models')
REP_DIR       = os.path.join(ROOT_DIR, 'reports')
FIG_DIR       = os.path.join(REP_DIR,  'figures')

for d in [DATA_DIR, RAW_DATA_DIR, PROC_DATA_DIR, FEAT_DIR, MOD_DIR, REP_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42

print(f'📁 Project root    : {ROOT_DIR}')
print(f'   data/raw/       : {RAW_DATA_DIR}')
print(f'   data/processed/ : {PROC_DATA_DIR}')
print(f'   features/       : {FEAT_DIR}')
print(f'   models/         : {MOD_DIR}')
print(f'✅ Import thư viện hoàn tất!')

## 2. 📥 Tải Dataset
> Nếu đã chạy `01_EDA` thì file đã có trong `data/raw/`. Nếu chưa, notebook này sẽ tự tải.

In [ ]:
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

TRAIN_URL  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
TEST_URL   = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test'
TRAIN_PATH = os.path.join(RAW_DATA_DIR, 'adult_train.csv')
TEST_PATH  = os.path.join(RAW_DATA_DIR, 'adult_test.csv')

if not os.path.exists(TRAIN_PATH):
    print('⏬ Đang tải dữ liệu từ UCI...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    urllib.request.urlretrieve(TEST_URL,  TEST_PATH)
    print('✅ Tải xong!')
else:
    print(f'✅ Dữ liệu đã có sẵn: {TRAIN_PATH}')

df_train = pd.read_csv(TRAIN_PATH, names=COLUMNS, sep=r',\s*', engine='python', na_values='?')
df_test  = pd.read_csv(TEST_PATH,  names=COLUMNS, sep=r',\s*', engine='python', na_values='?', skiprows=1)
df_full  = pd.concat([df_train, df_test], ignore_index=True)
df_full['income'] = df_full['income'].str.replace('.', '', regex=False).str.strip()
df = df_full.drop_duplicates().reset_index(drop=True)

print(f'✅ Loaded: {df.shape[0]:,} mẫu × {df.shape[1]} cột')
print(f'Missing values: {df.isnull().sum().sum()} trong {df.isnull().any().sum()} cột')

## 3. ✂️ Tách Features & Target + Train-Test Split

In [ ]:
numeric_features     = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical_features = ['workclass', 'education', 'marital_status', 'occupation',
                        'relationship', 'race', 'sex', 'native_country']

X = df.drop(columns=['income'])
y = (df['income'] == '>50K').astype(int)
target_pct = df['income'].value_counts(normalize=True) * 100

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Class ratio (train): >50K = {y_train.mean():.3f}')

# Lưu raw train/test CSV
X_train_save = X_train.copy(); X_train_save['income'] = y_train.map({0:'<=50K',1:'>50K'})
X_test_save  = X_test.copy();  X_test_save['income']  = y_test.map({0:'<=50K',1:'>50K'})
X_train.to_csv(os.path.join(PROC_DATA_DIR, 'X_train_raw.csv'), index=False)
X_test.to_csv( os.path.join(PROC_DATA_DIR, 'X_test_raw.csv'),  index=False)
y_train.to_csv(os.path.join(PROC_DATA_DIR, 'y_train.csv'), index=False)
y_test.to_csv( os.path.join(PROC_DATA_DIR, 'y_test.csv'),  index=False)
print('✅ Lưu raw CSV xong!')

## 4. 🛠️ Định Nghĩa Preprocessing Pipeline (Có Thể Cấu Hình)

In [ ]:
def build_preprocessor(impute_num='median', impute_cat='most_frequent',
                        encoding='onehot', scaling='standard'):
    """
    Xây dựng preprocessing pipeline có thể cấu hình.
    *** Phải giống hệt hàm trong main_notebook ***

    Parameters
    ----------
    impute_num  : 'mean' | 'median' | 'knn'
    impute_cat  : 'most_frequent' | 'constant'
    encoding    : 'onehot' | 'label'
    scaling     : 'standard' | 'minmax' | 'robust'
    """
    # ── Imputer ──
    if impute_num == 'knn':
        num_imputer = KNNImputer(n_neighbors=5)
    else:
        num_imputer = SimpleImputer(strategy=impute_num)
    cat_imputer = SimpleImputer(strategy=impute_cat,
                                fill_value='Unknown' if impute_cat == 'constant' else None)

    # ── Scaler ──
    if scaling == 'standard':
        scaler = StandardScaler()
    elif scaling == 'minmax':
        scaler = MinMaxScaler(feature_range=(0, 1))
    else:
        scaler = RobustScaler()

    # ── Encoder ──
    if encoding == 'onehot':
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    elif encoding == 'label':
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    else:
        # target encoding — fallback về onehot (giống main_notebook)
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    numeric_pipeline     = Pipeline([('imputer', num_imputer), ('scaler', scaler)])
    categorical_pipeline = Pipeline([('imputer', cat_imputer), ('encoder', encoder)])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline,     numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])
    return preprocessor

print('✅ build_preprocessor() sẵn sàng!')
print('   Tham số: impute_num (mean|median|knn) | impute_cat (most_frequent|constant)')
print('            encoding (onehot|label) | scaling (standard|minmax|robust)')

## 5. 📊 So Sánh Các Cấu Hình Preprocessing
Dùng Logistic Regression làm "probe" nhanh để chọn config tốt nhất.

In [ ]:
PREP_CONFIGS = {
    'Mean + OneHot + Standard' : dict(impute_num='mean',   encoding='onehot', scaling='standard'),
    'Median + OneHot + MinMax' : dict(impute_num='median', encoding='onehot', scaling='minmax'),
    'Median + Label + Robust'  : dict(impute_num='median', encoding='label',  scaling='robust'),
    'KNN + OneHot + Standard'  : dict(impute_num='knn',    encoding='onehot', scaling='standard'),
}

lr_probe = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, class_weight='balanced')

prep_results = {}
for name, cfg in PREP_CONFIGS.items():
    prep = build_preprocessor(**cfg)
    X_tr_p = prep.fit_transform(X_train, y_train)
    X_te_p = prep.transform(X_test)
    lr_probe.fit(X_tr_p, y_train)
    acc = lr_probe.score(X_te_p, y_test)
    prep_results[name] = {'accuracy': acc, 'n_features': X_tr_p.shape[1]}
    print(f'[{name:<30}]  accuracy={acc:.4f}  features={X_tr_p.shape[1]}')

best_cfg_name = max(prep_results, key=lambda k: prep_results[k]['accuracy'])
print(f'\n✅ Cấu hình tốt nhất: {best_cfg_name}  (accuracy={prep_results[best_cfg_name]["accuracy"]:.4f})')

In [ ]:
# Biểu đồ so sánh preprocessing configs
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(prep_results.keys())
accs   = [prep_results[n]['accuracy'] for n in names]
colors = ['#27ae60' if n == best_cfg_name else '#3498db' for n in names]
bars   = ax.barh(names, accs, color=colors, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, accs):
    ax.text(v + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{v:.4f}', va='center', fontsize=10)
ax.set_xlabel('Accuracy (LR probe)')
ax.set_title('So sánh Preprocessing Configs — LR Probe Accuracy', fontsize=13, fontweight='bold')
ax.set_xlim(0.7, 0.9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'preprocessing_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. ✅ Áp Dụng Best Config & Lưu Features

In [ ]:
# Áp dụng cấu hình tốt nhất
preprocessor = build_preprocessor(**PREP_CONFIGS[best_cfg_name])

X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'Shape sau preprocessing — Train: {X_train_proc.shape}, Test: {X_test_proc.shape}')

# Lấy tên feature
try:
    ohe_names  = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
    feat_names = list(numeric_features) + list(ohe_names)
except Exception:
    feat_names = [f'f{i}' for i in range(X_train_proc.shape[1])]

print(f'Total features: {len(feat_names)}')

# Lưu preprocessed features
np.save(os.path.join(FEAT_DIR, 'X_train_preprocessed.npy'), X_train_proc)
np.save(os.path.join(FEAT_DIR, 'X_test_preprocessed.npy'),  X_test_proc)
np.save(os.path.join(FEAT_DIR, 'y_train.npy'), y_train.values)
np.save(os.path.join(FEAT_DIR, 'y_test.npy'),  y_test.values)

# Lưu preprocessor
joblib.dump(preprocessor, os.path.join(MOD_DIR, 'preprocessor.pkl'))

# Lưu feature names
with open(os.path.join(FEAT_DIR, 'feature_names.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(feat_names))

print('✅ Đã lưu features và preprocessor!')

## 7. 📉 PCA — Giảm Chiều Dữ Liệu

In [ ]:
# Thử các ngưỡng PCA
for vr in [0.90, 0.95, 0.99]:
    pca_tmp = PCA(n_components=vr, random_state=RANDOM_STATE)
    n = pca_tmp.fit(X_train_proc).n_components_
    print(f'PCA {vr*100:.0f}%: {X_train_proc.shape[1]} → {n} components')

# Dùng PCA 95%
pca_95 = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca_95.fit_transform(X_train_proc)
X_test_pca  = pca_95.transform(X_test_proc)
print(f'\n✅ Dùng PCA 95%: {X_train_proc.shape[1]} → {X_train_pca.shape[1]} features')

# Scree plot
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_proc)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_) * 100
fig, ax  = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, 'b-o', markersize=3)
ax.axhline(95, color='red',    linestyle='--', label='95% threshold')
ax.axhline(99, color='orange', linestyle='--', label='99% threshold')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('PCA Scree Plot', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pca_scree.png'), dpi=150, bbox_inches='tight')
plt.show()

# Lưu PCA features + model
np.save(os.path.join(FEAT_DIR, 'X_train_pca.npy'), X_train_pca)
np.save(os.path.join(FEAT_DIR, 'X_test_pca.npy'),  X_test_pca)
joblib.dump(pca_95, os.path.join(MOD_DIR, 'pca_95.pkl'))
print('✅ Lưu PCA features và model!')

## 8. 📊 Visualize Trước / Sau Preprocessing

### 8.1. 🌲 Feature Importance (Random Forest Probe)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_probe = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                  n_jobs=-1, class_weight='balanced')
rf_probe.fit(X_train_proc, y_train)

# Lấy tên features
try:
    ohe_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
    all_names = numeric_features + list(ohe_names)
except Exception:
    all_names = [f'feature_{i}' for i in range(X_train_proc.shape[1])]

importances = rf_probe.feature_importances_
top_n       = min(20, len(all_names))
indices     = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(range(top_n), importances[indices], color='teal', alpha=0.8, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([all_names[i] for i in indices], fontsize=9)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n→ Top 5 features quan trọng nhất:')
for i in indices[:5]:
    print(f'   {all_names[i]}: {importances[i]:.4f}')

### 8.1. Visulize Most Important Feature Before and After


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X_train['age'].dropna(), bins=35, color='salmon', edgecolor='white', alpha=0.8)
axes[0].set_title('Trước Scaling: age', fontweight='bold')
axes[0].set_xlabel('age (original)')

axes[1].hist(X_train_proc[:, 0], bins=35, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Sau Standard Scaling: age', fontweight='bold')
axes[1].set_xlabel('age (scaled)')

plt.suptitle('Before vs After Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'before_after_scaling.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. 📋 Tổng Kết Preprocessing

In [ ]:
print('=' * 60)
print('TỔNG KẾT PREPROCESSING')
print('=' * 60)
print(f'Best config            : {best_cfg_name}')
print(f'Features gốc           : {X_train.shape[1]}')
print(f'Features sau encoding  : {X_train_proc.shape[1]}')
print(f'Features sau PCA 95%   : {X_train_pca.shape[1]}')
print()
print('📦 Files đã lưu:')
print(f'   {PROC_DATA_DIR}/X_train_raw.csv | X_test_raw.csv | y_train.csv | y_test.csv')
print(f'   {FEAT_DIR}/X_train_preprocessed.npy | X_test_preprocessed.npy')
print(f'   {FEAT_DIR}/X_train_pca.npy | X_test_pca.npy | y_train.npy | y_test.npy')
print(f'   {FEAT_DIR}/feature_names.txt')
print(f'   {MOD_DIR}/preprocessor.pkl | pca_95.pkl')
print()
print('✅ Preprocessing hoàn thành! Chạy 03_Traditional_ML.ipynb tiếp theo.')